<a href="https://colab.research.google.com/github/RitaLB/Stonelab-Churn-Clientes-AI/blob/main/notebooks/stone_churn_clients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pré processamento

## **Carregamento dos dados e pré processamento**

In [2]:
import urllib.request, csv, io
import pandas as pd, numpy as np

URL = "https://raw.githubusercontent.com/RitaLB/Stonelab-Churn-Clientes-AI/main/data/dataset-churn.csv"

# Dataset original.
raw = list(csv.reader(io.StringIO(urllib.request.urlopen(URL).read().decode("utf-8"))))
header, linhas = raw[0], raw[1:]
n_cols = len(header)
print(n_cols)

15


Upload Schema para tipagem das colunas - Segundo Dicionário de dados

In [3]:
import json
SCHEMA_URL = "https://raw.githubusercontent.com/RitaLB/Stonelab-Churn-Clientes-AI/main/data/schema_churn.json"
schema = json.loads(urllib.request.urlopen(SCHEMA_URL).read().decode("utf-8"))

Diagnostico de corretude do database

In [4]:
def validar(linhas, header, schema):
    """Valida cada célula contra o schema (fonte da verdade do dicionário).
    Retorna {num_linha: {coluna: motivo}} contendo apenas as violações."""

    def ok_tipo(v, t):
        # int: apenas dígitos (aceita sinal negativo). float: converte para número.
        # categorica/string/binaria não checam tipo aqui — são validadas pelo domínio.
        if t == "int":   return v.lstrip("-").isdigit()
        if t == "float": return pd.notna(pd.to_numeric([v], errors="coerce")[0])
        return True

    def ok_dominio(v, d):
        if not d:                      # coluna sem domínio definido (ex: id)
            return True
        if "valores" in d:             # categórica: valor deve estar na lista permitida
            return v in d["valores"]
        n = pd.to_numeric([v], errors="coerce")[0]   # numérica: checa faixa min/max
        if pd.isna(n):
            return False
        return ("min" not in d or n >= d["min"]) and ("max" not in d or n <= d["max"])

    erros = {}
    for i, r in enumerate(linhas, 1):
        # 1) Falha estrutural: nº de campos diferente do cabeçalho.
        #    Impede alinhar coluna↔valor, então a linha é reportada sem checagem célula a célula.
        if len(r) != len(header):
            erros[i] = {"_estrutural": f"{len(r)} campos (esperado {len(header)})"}
            continue

        # 2) Linha alinhada: valida cada célula na ordem tipo -> domínio.
        linha_erros = {}
        for col, v in zip(header, r):
            if v == "":                                      # valor ausente
                linha_erros[col] = "vazio"
            elif not ok_tipo(v, schema[col]["tipo"]):        # tipo incompatível
                linha_erros[col] = f"tipo!={schema[col]['tipo']}"
            elif not ok_dominio(v, schema[col]["dominio"]):  # fora do domínio permitido
                linha_erros[col] = "fora do dominio"
        if linha_erros:
            erros[i] = linha_erros

    return erros


erros = validar(linhas, header, schema)
erros

{5: {'plano': 'vazio'},
 17: {'nps_ultimo': 'vazio'},
 27: {'_estrutural': '16 campos (esperado 15)'}}

Correção erro estrutural

In [5]:
from itertools import combinations

def corrigir_estrutural(r, header, schema):
    """Corrige linha com campos a mais. Só remove campos VAZIOS, e aceita a
    correção apenas se a linha resultante for válida no schema. Isso distingue
    vazio espúrio (vírgula extra) de vazio legítimo (faltante real)."""
    n = len(header)

    # Linha já alinhada: nada a corrigir.
    if len(r) == n:
        return r, None

    excesso = len(r) - n                              # quantos campos sobram
    pos_vazias = [k for k, v in enumerate(r) if v == ""]  # posições vazias candidatas

    # Se os campos extras não são vazios, não é vírgula espúria — não corrigimos.
    if len(pos_vazias) < excesso:
        return None, "campos extras nao sao vazios"

    # Testa cada combinação de campos vazios a remover.
    # A remoção correta realinha os valores nas colunas certas; a errada
    # desloca valores e gera erro de tipo/domínio na validação.
    for pos in combinations(pos_vazias, excesso):
        cand = [v for k, v in enumerate(r) if k not in pos]
        erros = validar([cand], header, schema).get(1, {})

        # Aceita só se os erros restantes forem apenas "vazio" (faltantes
        # legítimos, tratados na imputação). Qualquer erro de tipo/domínio
        # significa desalinhamento → candidata rejeitada.
        if all(m == "vazio" for m in erros.values()):
            return cand, None

    return None, "nenhuma remocao produz linha valida"

# Aplica correção estrutural
linhas_corrigidas, descartes = [], []
for i, r in enumerate(linhas, 1):
    c, motivo = corrigir_estrutural(r, header, schema)
    linhas_corrigidas.append(c) if c is not None else descartes.append((i, motivo))

# Converte "" -> None (faltantes legítimos, tratados na imputação)
dados = [[None if v == "" else v for v in r] for r in linhas_corrigidas]

# Confirmação: nenhum erro estrutural/tipo/domínio deve restar (apenas "vazio")
erros_finais = validar(linhas_corrigidas, header, schema)
erros_finais

{5: {'plano': 'vazio'}, 17: {'nps_ultimo': 'vazio'}}

Savar tabela pré processada

In [6]:
import csv

with open("dataset-churn-corrigido.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(dados)

# 4.1 Representação do Conhecimento

**Fonte da verdade para tipos e domínios:** dicionário de dados (schema_churn.json).

**Transformações pré-EDA** (preparam o dado para ser analisado, sem alterar distribuições):



*   **Descarte de cliente_id** — remoção de coluna. Identificador único, sem poder preditivo.

*   **Tipagem conforme schema** (Int64, float, string) : conversão de tipo. Int64 preserva NaN em colunas inteiras.


**Transformações pós-EDA** (aplicadas após entender o dado, alimentam o modelo):


*   **Imputação** : preenche faltantes: mediana (numéricas, robusta a outliers), moda (categóricas).

*   **One-Hot Encoding** : converte categórica em colunas binárias (0/1) por categoria. Evita impor ordem inexistente.

* **Padronização** (z-score) : reescala numéricas para média 0 e desvio 1. Evita que features de escala grande (TPV) dominem as de escala pequena (NPS).




Transformaçõe pré EDA:

In [7]:
import pandas as pd

# 1. DataFrame tipado conforme schema
df = pd.DataFrame(dados, columns=header)
for col, spec in schema.items():
    if spec["tipo"] in ("int", "binaria"):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    elif spec["tipo"] == "float":
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = df[col].astype("string")

# 2. Descarte de identificador
df = df.drop(columns=["cliente_id"])

# 3. Separação por papel (imputação/encoding aplicados após a EDA)
alvo = "churn"
num_cols = [c for c, s in schema.items()
            if s["tipo"] in ("int", "float") and s.get("usar_como_feature")]
cat_cols = [c for c, s in schema.items()
            if s["tipo"] == "categorica" and s.get("usar_como_feature")]

In [ ]:
# Teste se transformação deu certo

In [12]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum()[lambda s: s > 0])

(30, 14)
tpv_medio_mensal                 int64
variacao_tpv_3m                float64
dias_sem_transacao               Int64
tempo_cliente_meses              Int64
num_produtos_ativos              Int64
ticket_medio                   float64
num_chamados_suporte             Int64
nps_ultimo                       Int64
segmento                string[python]
regiao                  string[python]
tem_maquininha          string[python]
plano                   string[python]
usa_app_mobile          string[python]
churn                            Int64
dtype: object
nps_ultimo    1
plano         1
dtype: int64


#4.2 — Análise Exploratória (EDA)

- Distribuições, correlações, outliers, desbalanceamento de classes.
- Visualizações que revelem padrões relevantes para o problema.
- Hipóteses sobre quais features devem ser mais preditivas.

Entregável: Visualizações comentadas no notebook.


In [13]:
# Faltantes por coluna (só as que têm ao menos um) -> células vazias. Modelos não aceitam vazio; é preciso preencher (imputar).
print(df.isna().sum()[lambda s: s > 0])

# Proporção das classes do alvo -> mede desbalanceamento.
# Se desbalanceado: usar F1/AUC + class_weight="balanced" (acurácia enganaria).
# Se equilibrado: acurácia basta.
print(df["churn"].value_counts(normalize=True))

nps_ultimo    1
plano         1
dtype: int64
churn
0    0.633333
1    0.366667
Name: proportion, dtype: Float64
